# 12 — Complaint Escalation & Priority Prediction
Predicting which complaints require urgent escalation or monetary relief based on narrative complexity, category, and detected emotion.

In [1]:
import pandas as pd, numpy as np
import plotly.express as px, plotly.graph_objects as go
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix, roc_curve
import joblib, os
import warnings; warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)

In [2]:
# ── Load and Target Engineering ──
df = pd.read_csv('../../data/processed/complaints_with_nlp.csv')
threshold_wc = df['narrative_word_count'].quantile(0.85)

df['escalation_flag'] = (
    (df['company_response'] == 'Closed with monetary relief') |
    (df['timely'] == 'No') |
    (df['llm_emotion'].isin(['Anger', 'Legal Threat'])) |
    (df['narrative_word_count'] > threshold_wc)
).astype(int)

print(f"Escalation flag distribution:")
print(df['escalation_flag'].value_counts())
print(f"Escalation rate: {df['escalation_flag'].mean()*100:.2f}%")

Escalation flag distribution:
escalation_flag
1    663
0    337
Name: count, dtype: int64
Escalation rate: 66.30%


In [3]:
# ── Feature Engineering ──
le_cat = LabelEncoder()
le_emo = LabelEncoder()
le_prod = LabelEncoder()

df['category_enc'] = le_cat.fit_transform(df['llm_category'].fillna('Other'))
df['emotion_enc'] = le_emo.fit_transform(df['llm_emotion'].fillna('Neutral'))
df['product_enc'] = le_prod.fit_transform(df['product'].fillna('Unknown'))
df['timely_binary'] = (df['timely'] == 'No').astype(int)

features = ['narrative_word_count','category_enc','emotion_enc', 'product_enc','timely_binary']

X = df[features]
y = df['escalation_flag']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Train: (800, 5), Test: (200, 5)


In [4]:
# ── Train Models and Evaluate ──
pos_w = (y_train==0).sum() / max((y_train==1).sum(), 1)

rf_esc = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1)
rf_esc.fit(X_train, y_train)
rf_prob = rf_esc.predict_proba(X_test)[:,1]
rf_pred = rf_esc.predict(X_test)

xgb_esc = XGBClassifier(scale_pos_weight=pos_w, random_state=42, verbosity=0, n_estimators=200)
xgb_esc.fit(X_train, y_train)
xgb_prob = xgb_esc.predict_proba(X_test)[:,1]
xgb_pred = xgb_esc.predict(X_test)

print("=== ESCALATION MODEL RESULTS (REAL) ===")
for name, prob, pred in [
    ('Random Forest', rf_prob, rf_pred),
    ('XGBoost', xgb_prob, xgb_pred)
]:
    print(f"\n{name}:")
    print(f"  Accuracy: {accuracy_score(y_test, pred):.4f}")
    print(f"  F1: {f1_score(y_test, pred):.4f}")
    print(f"  ROC-AUC: {roc_auc_score(y_test, prob):.4f}")
    print(classification_report(y_test, pred, target_names=['Standard','Escalated']))

=== ESCALATION MODEL RESULTS (REAL) ===

Random Forest:
  Accuracy: 0.8950
  F1: 0.9195
  ROC-AUC: 0.9538
              precision    recall  f1-score   support

    Standard       0.82      0.88      0.85        67
   Escalated       0.94      0.90      0.92       133

    accuracy                           0.90       200
   macro avg       0.88      0.89      0.88       200
weighted avg       0.90      0.90      0.90       200


XGBoost:
  Accuracy: 0.8850
  F1: 0.9112
  ROC-AUC: 0.9523
              precision    recall  f1-score   support

    Standard       0.80      0.88      0.84        67
   Escalated       0.94      0.89      0.91       133

    accuracy                           0.89       200
   macro avg       0.87      0.88      0.87       200
weighted avg       0.89      0.89      0.89       200



In [5]:
# ── Generate Escalation Probabilities & Visualizations ──
df['escalation_probability'] = xgb_esc.predict_proba(X)[:,1]
df['priority_level'] = pd.cut(
    df['escalation_probability'],
    bins=[-0.001, 0.50, 0.80, 0.90, 1.001],
    labels=['Standard','High','Critical','Urgent']
)
print("Real priority distribution:")
print(df['priority_level'].value_counts())

# Plot 1: Priority Level Distribution
fig1 = px.pie(df, names='priority_level', title="Complaint Priority Tier Distribution", hole=0.4)
fig1.show()

# Plot 2: ROC Curve
fpr, tpr, _ = roc_curve(y_test, xgb_prob)
fig2 = px.line(x=fpr, y=tpr, title=f"XGBoost Escalation ROC Curve (AUC={roc_auc_score(y_test, xgb_prob):.3f})")
fig2.add_shape(type='line', line=dict(dash='dash'), x0=0, x1=1, y0=0, y1=1)
fig2.update_layout(template='plotly_white', xaxis_title='FPR', yaxis_title='TPR')
fig2.show()

# Plot 3: Feature Importance
fi = pd.Series(xgb_esc.feature_importances_, index=features).sort_values()
fig3 = px.bar(x=fi.values, y=fi.index, orientation='h', title="Feature Importance (Escalation Model)")
fig3.show()

# Plot 4: Escalation Probability by Emotion
fig4 = px.box(df, x='llm_emotion', y='escalation_probability', color='llm_emotion', title="Escalation Probability by Detected Emotion")
fig4.show()

Real priority distribution:
priority_level
Urgent      602
Standard    352
High         27
Critical     19
Name: count, dtype: int64


In [6]:
# ── Save ──
os.makedirs('../../models/escalation', exist_ok=True)
joblib.dump(xgb_esc, '../../models/escalation/xgboost_escalation.pkl')
df.to_csv('../../data/processed/complaints_with_escalation.csv', index=False)
print("=== ESCALATION PIPELINE COMPLETE AND SAVED ===")

=== ESCALATION PIPELINE COMPLETE AND SAVED ===
